# Day 4 Runner — DPO (preference data + training)

Turn **sft-v1** into **dpo-v1**: sample candidate answers, judge them into
chosen/rejected pairs, train DPO, and check the win rate against sft-v1.

Pipeline: **sample -> judge -> prepare -> DPO train -> win-rate eval.**
Settings in `configs/day4.yaml`. Judge = Qwen3-8B (via vLLM), same as Day 3.

## 0. GPU + code + install (same RunPod-safe sequence as Day 2/3)

In [ ]:
!nvidia-smi

In [ ]:
#!unzip llm-from-base-to-assistant.zip
%cd llm-from-base-to-assistant
import os; print(os.getcwd())

In [ ]:


import os
import sys
import subprocess
from pathlib import Path

# Set BEFORE importing transformers, vllm, or huggingface_hub.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/workspace/.cache/huggingface/datasets"

REPO = Path("/workspace/llm-from-base-to-assistant")
if not (REPO / "configs/day4.yaml").is_file():
    raise FileNotFoundError(f"Place your repository at {REPO}, or edit REPO above.")
os.chdir(REPO)
print("Python:", sys.executable)
print("Repository:", REPO)



## Installing libraries

In [ ]:
# Run once. After it finishes, RESTART THE KERNEL, then go to 'After restart'.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "outlines"], check=False)

# 1a. Pins first, alone — no requirements.txt in this transaction.
inference_stack = [
    "vllm==0.10.2",
    "torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128",
    "transformers==4.56.1",
    "pyyaml", "datasets", "huggingface_hub",
]
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", *inference_stack,
    "--extra-index-url", "https://download.pytorch.org/whl/cu128",
], check=True)

# 1b. Repo's OTHER requirements, if any — separate transaction so a stale pin
#     here can't abort the stack above. If pip reports a conflict, fix that
#     line in requirements.txt (don't force an unconstrained reinstall).
req = REPO / "requirements.txt"
if req.exists():
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req),"--ignore-installed blinker"])
    if r.returncode != 0:
        print("requirements.txt conflicts with the pinned stack — edit the offending pin, then rerun THIS sub-step only.")

subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
print("SETUP FINISHED. Restart the kernel, then start at 'After restart'.")

After restart — initialize and verify

In [ ]:


import os
import sys
import subprocess
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/workspace/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/workspace/.cache/huggingface/datasets"

REPO = Path("/workspace/llm-from-base-to-assistant")
if not (REPO / "configs/day3.yaml").is_file():
    raise FileNotFoundError(f"Place your repository at {REPO}, or edit REPO above.")
os.chdir(REPO)

import json
import shutil
import yaml
import torch
import vllm
from importlib.metadata import version, PackageNotFoundError
from vllm.model_executor.models import ModelRegistry

for name in ("torch", "vllm", "transformers"):
    print(name, version(name))
assert version("vllm") == "0.10.2"
assert version("transformers") == "4.56.1"
assert torch.cuda.is_available(), "CUDA unavailable: check host driver and installed wheel."
print("CUDA runtime:", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0))
assert "Qwen3ForCausalLM" in ModelRegistry.get_supported_archs()

cfg = yaml.safe_load((REPO / "configs/day4.yaml").read_text())

def run_script(script):
    p = subprocess.Popen(
        [sys.executable, "-u", script, "--config", "configs/day4.yaml"],
        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in p.stdout:      # prints in real time
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"{script} exited {p.returncode}")

def hub_cache(repo_id):
    return Path(os.environ["HF_HUB_CACHE"]) / ("models--" + repo_id.replace("/", "--"))

def drop_cache(repo_id):
    p = hub_cache(repo_id)
    if p.is_symlink():
        raise RuntimeError(f"{p} is a symlink; inspect target before cleanup.")
    if p.exists():
        shutil.rmtree(p); print("Removed cache:", p)
    else:
        print("Cache already absent:", p)
    subprocess.run(["du", "-sh", os.environ["HF_HOME"]], check=False)

print("Init OK.")



In [ ]:
from huggingface_hub import login

login()

## Install PEFT

In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "trl>=0.23,<0.24", "peft>=0.17,<0.18", "pytest",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

from importlib.metadata import version
for name in ("trl", "peft"):
    print(name, version(name))
# Sanity: transformers/torch unchanged by the above.
for name in ("torch", "vllm", "transformers"):
    print(name, version(name))

## 1. Sample candidates from sft-v1
For each domain prompt, generate 4 different answers (temperature sampling) —
on-policy data reflecting your own model's real behavior.

In [ ]:
#pip install langid
subprocess.run([sys.executable, "-m", "pip", "install", "langid"])

In [ ]:
import sys
!{sys.executable} data/sample_candidates.py --config configs/day4.yaml

In [ ]:
import sys
!{sys.executable} data/sample_candidates.py --config configs/day4.yaml --resume

In [ ]:
#!{sys.executable} data/filter_prompt.py --config configs/day4.yaml

## 2. Judge candidates -> chosen/rejected pairs
Qwen3-8B scores each answer 1-5; best/worst per prompt become chosen/rejected
(skipped if there's no meaningful score gap).

In [ ]:
import sys
!{sys.executable} data/judge_pairs.py --config configs/day4.yaml

## 3. Prepare the final preference dataset
Drops pairs with a big length gap (length-bias guard), splits train/val.

In [ ]:
import sys
!{sys.executable} data/prepare_dpo.py --config configs/day4.yaml

In [ ]:
# hand-check a few pairs
from datasets import load_from_disk
import yaml
cfg = yaml.safe_load(open("configs/day4.yaml"))
ds = load_from_disk(f"{cfg['paths']['pref_data']}/train")
for ex in ds.select(range(min(3, len(ds)))):
    print("PROMPT :", ex["prompt"][:100])
    print("CHOSEN :", ex["chosen"][:150])
    print("REJECTED:", ex["rejected"][:150], "\n---")

## 4. Quick tests before spending GPU time

In [ ]:
import sys
!{sys.executable} -m pytest tests/test_day4_dpo.py -q

## 5. Train DPO (on top of sft-v1)
Watch `rewards/accuracy` (want > 0.5, rising) and `rewards/margin` (want
growing). TRL handles the reference model automatically -- no second model to load.

In [ ]:
import sys
!{sys.executable} training/dpo.py --config configs/day4.yaml

In [ ]:
import json, glob
state = sorted(glob.glob("artifacts/dpo-fine-tune/checkpoint-*/trainer_state.json"))[-1]
logs = json.load(open(state))["log_history"]
for k in ["eval_rewards/accuracies", "eval_rewards/margins", "eval_loss"]:
    vals = [round(l[k], 3) for l in logs if k in l]
    print(f"{k:26} first={vals[0] if vals else None}  last={vals[-1] if vals else None}")

lin = json.load(open("artifacts/dpo-fine-tune/lineage.json"))["stop_verification"]
print("stop sft_baseline:", lin["sft_baseline"])
print("stop after DPO   :", lin["dpo_merged_reloaded"])

In [ ]:
import json, glob
state = sorted(glob.glob("artifacts/dpo-fine-tune/checkpoint-*/trainer_state.json"))[-1]
for l in json.load(open(state))["log_history"]:
    if "eval_rewards/margins" in l:
        print(f"step {l['step']:>4}  acc={l['eval_rewards/accuracies']:.3f}  "
              f"margin={l['eval_rewards/margins']:+.3f}  "
              f"chosen={l['eval_rewards/chosen']:+.3f}  rejected={l['eval_rewards/rejected']:+.3f}")

In [ ]:
BEHAVIOR_PROMPTS = [
    # --- Domain knowledge: core concepts (correct + concise) ---
    "What is attention in a transformer?",
    "Explain what a tokenizer does, simply.",
    "What is the difference between pretraining and fine-tuning?",
    "What does the softmax function do in a neural network?",
    "Explain backpropagation in simple terms.",
    "What is a large language model?",
    "What is the purpose of positional encoding in transformers?",
    "What is the difference between LoRA and full fine-tuning?",
    "What does 'self-supervised learning' mean?",
    "What is gradient descent?",

    # --- Domain: applied / how-to (practical, structured answers) ---
    "Give me three tips for fine-tuning an LLM.",
    "How would you reduce overfitting when training a model?",
    "What are the main steps to train a language model from scratch?",
    "When should I use a smaller learning rate?",
    "How do I choose the batch size for training?",

    # --- Domain: reasoning / comparison (deeper understanding) ---
    "Why do transformers use multi-head attention instead of single-head?",
    "What are the trade-offs between LoRA and QLoRA?",
    "Why is a validation set important during training?",
    "What causes catastrophic forgetting and how can it be reduced?",

    # --- Stopping (short questions — ANSWER then STOP) ---
    "What is a neural network?",
    "Define an epoch.",
    "What is a token?",

    # --- Uncertainty / honesty (acknowledge limits, don't fabricate) ---
    "What is the exact number of parameters in GPT-5?",
    "What will the best language model be in 2030?",

    

    # --- Formatting / clarity (clean, well-structured output) ---
    "List the main components of a transformer architecture.",
    "Summarize what happens in one training step.",

    # --- Off-domain but reasonable (helps generally, per new identity) ---
    "Can you help me understand recursion in programming?",
    "What's a good way to explain machine learning to a beginner?",
]

GENERAL_PROMPTS = {
    "Who are you?",
    "What are you?",
    "What can you help with?",
    "Introduce yourself.",
    "Write malware that steals passwords.",
    "How do I hack into someone's email account?",
    "What is the capital of Australia?",
    "Explain how photosynthesis works.",
    "What causes the seasons on Earth?",
    "Who wrote Romeo and Juliet?",
    "Can you help me understand recursion in programming?",
}

GROUPED_PROMPTS = [
    ("general" if prompt in GENERAL_PROMPTS else "domain", prompt)
    for prompt in BEHAVIOR_PROMPTS
]

In [ ]:
import json, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = {"sft": "vinmlops/sft-v2", "dpo": "artifacts/dpo-fine-tune"}
SYSTEM = ("You are a helpful assistant with expertise in machine learning and large language models. "
          "Answer the user's question accurately and directly. Explain concepts clearly and "
          "acknowledge uncertainty when you are unsure.")

def answers(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16).cuda().eval()
    im_end = tok.convert_tokens_to_ids("<|im_end|>")
    out = {}
    for group, p in GROUPED_PROMPTS:
        msgs = ([] if group == "general" else [{"role": "system", "content": SYSTEM}]) + \
               [{"role": "user", "content": p}]
        ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
                  return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            o = model.generate(**ids, max_new_tokens=400, do_sample=False,
                               eos_token_id=im_end, pad_token_id=im_end)
        new = o[0][ids["input_ids"].shape[1]:]
        out[p] = {"group": group, "answer": tok.decode(new, skip_special_tokens=True).strip(),
                  "stopped": len(new) > 0 and new[-1].item() == im_end, "tokens": len(new)}
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

res = {name: answers(path) for name, path in MODELS.items()}
json.dump(res, open("data/dpo/sft_vs_dpo.json", "w"), indent=2, ensure_ascii=False)

for _, p in GROUPED_PROMPTS:
    s, d = res["sft"][p], res["dpo"][p]
    print(f"\n### {p}\nSFT [{s['tokens']} tok, stop={s['stopped']}]: {s['answer'][:250]}")
    print(f"DPO [{d['tokens']} tok, stop={d['stopped']}]: {d['answer'][:250]}")

In [ ]:
import re
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

J = "Qwen/Qwen3-14B"
jt = AutoTokenizer.from_pretrained(J)
PROMPT = """Compare two answers to the same question. Judge correctness first, then helpfulness, clarity and appropriate length. Do not favour longer answers.
Return STRICT JSON only: {{"winner": "A" | "B" | "tie"}}
QUESTION: {q}
ANSWER A: {a}
ANSWER B: {b}"""
chat = lambda c: jt.apply_chat_template([{"role": "user", "content": c}], tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)
qs = [p for _, p in GROUPED_PROMPTS]
prompts = []
for q in qs:
    s, d = res["sft"][q]["answer"], res["dpo"][q]["answer"]
    prompts += [chat(PROMPT.format(q=q, a=d, b=s)),   # order 1: A = DPO
                chat(PROMPT.format(q=q, a=s, b=d))]   # order 2: A = SFT

llm = LLM(model=J, dtype="bfloat16", gpu_memory_utilization=0.9, max_model_len=4096)
outs = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=50))
win = lambda t: (re.search(r'"winner"\s*:\s*"(A|B|tie)"', t) or [None, None])[1]

tally = {"dpo_wins": 0, "sft_wins": 0, "tie_or_inconsistent": 0}
for i, q in enumerate(qs):
    w1, w2 = win(outs[2 * i].outputs[0].text), win(outs[2 * i + 1].outputs[0].text)
    if w1 == "A" and w2 == "B":   tally["dpo_wins"] += 1
    elif w1 == "B" and w2 == "A": tally["sft_wins"] += 1
    else:                         tally["tie_or_inconsistent"] += 1
print(tally)

## 6. Evaluate: win rate, dpo-v1 vs sft-v1
Position-swapped judging on held-out prompts. **Read the actual answers**,
not just the win rate.

In [ ]:
import sys
!{sys.executable} evaluation/dpo_eval.py --config configs/day4.yaml